# Strict 5-fold OOF symptom classification and causal graph evaluation

This notebook runs the complete Text2DAG experiment in Google Colab:

1. build/reuse patient sentence mappings and ModernBERT patient embeddings;
2. train the **legacy reproduction classifier head** with **5%, 10%, 20%, and 100% of each outer 80% training pool**;
3. generate exactly one held-out OOF prediction per patient and report per-symptom and macro precision/recall/F1;
4. replace only the five symptom variables in SynSUM by OOF predictions;
5. run PC-learn once on each complete 10,000-patient reconstruction and once on the oracle data.

The supervised stage uses the legacy 768→256→5 linear head (no activation), unweighted BCE, AdamW at `3e-5`, no feature scaling, raw fever targets `0/1/2`, and a strict `>0.5` prediction threshold. The embeddings remain the existing ModernBERT patient features. Every graph condition uses **stable PC + G-square (`g_sq`) + alpha = 0.05**, `return_type='dag'`, and `DiscreteBayesianNetwork`. The old partly in-sample `all_dataset_predictions` files are never used.


## Before running

Select **Runtime → Change runtime type → GPU**. Put `SynSUM.csv` in `MyDrive/Text2DAG/inputs/`. The reference graph `expert_dag_adjacency.csv` is versioned with this repository and is loaded automatically.

- `SynSUM.csv` — semicolon-delimited source data with patient ID column `Unnamed: 0`;
- repository `expert_dag_adjacency.csv` — named binary reference adjacency: first column contains node names, remaining columns form a square matrix with row=source and column=target.

The notebook creates the sentence mapping and embeddings if they are not already present, then stores all experiment outputs persistently in Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Repository settings. The notebook and implementation are versioned together on main.
REPOSITORY_URL = 'https://github.com/sooncake/Text2DAG.git'
REPOSITORY_BRANCH = 'main'
PROJECT_DIR = Path('/content/Text2DAG')

# Persistent Google Drive paths.
DRIVE_ROOT = Path('/content/drive/MyDrive/Text2DAG')
INPUT_DIR = DRIVE_ROOT / 'inputs'
SOURCE_PATH = INPUT_DIR / 'SynSUM.csv'
REFERENCE_ADJACENCY_PATH = PROJECT_DIR / 'expert_dag_adjacency.csv'
MAPPING_OUTPUT_DIR = DRIVE_ROOT / 'preprocessing'
MAPPING_PATH = MAPPING_OUTPUT_DIR / 'gfs_sentence_mapping.csv'
EMBEDDING_OUTPUT_DIR = DRIVE_ROOT / 'generated_sentence_embeddings'
EMBEDDING_NPZ_PATH = EMBEDDING_OUTPUT_DIR / 'patient_embeddings_and_labels.npz'
OOF_OUTPUT_DIR = DRIVE_ROOT / 'oof_graph_experiment_legacy'

PATIENT_ID_COLUMN = 'Unnamed: 0'
RANDOM_SEED = 42
EXPECTED_PATIENTS = 10_000
EMBEDDING_BATCH_SIZE = 32
FORCE_REBUILD_MAPPING = False
FORCE_RECOMPUTE_EMBEDDINGS = False
FORCE_RERUN_OOF_EXPERIMENT = False

for directory in [INPUT_DIR, MAPPING_OUTPUT_DIR, EMBEDDING_OUTPUT_DIR, OOF_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Source:', SOURCE_PATH)
print('Reference DAG:', REFERENCE_ADJACENCY_PATH)
print('OOF outputs:', OOF_OUTPUT_DIR)


## Clone/update the implementation and install dependencies

The notebook orchestrates the repository modules instead of copying the classifier and graph implementations into notebook cells. This keeps Colab and command-line behavior identical.


In [ ]:
import subprocess
import sys

if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', REPOSITORY_BRANCH], check=True)
    subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', REPOSITORY_BRANCH],
        check=True,
    )
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPOSITORY_BRANCH, REPOSITORY_URL, str(PROJECT_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')],
    check=True,
)
current_branch = subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'branch', '--show-current'], text=True
).strip()
current_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
graph_source = (PROJECT_DIR / 'run_oof_graph_experiment.py').read_text(encoding='utf-8')
legacy_oof_source = (PROJECT_DIR / 'legacy_oof_symptom_classifier.py').read_text(encoding='utf-8')
if (
    'implementation: str = "pgmpy"' not in graph_source
    or 'return_type: str = "dag"' not in graph_source
    or '"--classifier-backend"' not in graph_source
    or 'class ClassifierConfig' not in legacy_oof_source
):
    raise RuntimeError(
        f'Checked out {current_branch}@{current_commit}, but the legacy OOF + pgmpy DAG implementation is missing.'
    )
print('Repository and dependencies are ready:', f'{current_branch}@{current_commit}')


In [ ]:
import json
import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError('A Colab GPU runtime is required. Select Runtime > Change runtime type > GPU.')
for required_path in [SOURCE_PATH, REFERENCE_ADJACENCY_PATH]:
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing required input: {required_path}')

source_preview = pd.read_csv(SOURCE_PATH, sep=';', nrows=5)
reference_preview = pd.read_csv(REFERENCE_ADJACENCY_PATH)
if PATIENT_ID_COLUMN not in source_preview.columns:
    raise ValueError(f'SynSUM is missing patient ID column {PATIENT_ID_COLUMN!r}.')
if reference_preview.shape[1] != reference_preview.shape[0] + 1:
    raise ValueError('Reference adjacency must have one name column plus a square matrix.')

reference_nodes = reference_preview.iloc[:, 0].astype(str).tolist()
missing_graph_columns = [name for name in reference_nodes if name not in source_preview.columns]
if missing_graph_columns:
    raise ValueError(
        'Reference graph node names must exactly match SynSUM columns; missing: '
        f'{missing_graph_columns}'
    )
if not {'dysp', 'cough', 'pain', 'fever', 'nasal'}.issubset(reference_nodes):
    raise ValueError('Reference graph must contain all five symptom nodes.')

print('GPU:', torch.cuda.get_device_name(0))
print('SynSUM columns:', source_preview.columns.tolist())
print('Reference graph nodes:', reference_nodes)
print('PASS: every reference node has an exact SynSUM column match.')


## 1. Build or reuse the patient sentence mapping


In [ ]:
if MAPPING_PATH.is_file() and not FORCE_REBUILD_MAPPING:
    print('Reusing:', MAPPING_PATH)
else:
    subprocess.run(
        [
            sys.executable, str(PROJECT_DIR / 'build_gfs_sentence_mapping.py'),
            '--input-path', str(SOURCE_PATH),
            '--output-dir', str(MAPPING_OUTPUT_DIR),
            '--id-column', PATIENT_ID_COLUMN,
        ],
        check=True,
    )
if not MAPPING_PATH.is_file():
    raise FileNotFoundError(f'Mapping was not created: {MAPPING_PATH}')


## 2. Build or reuse ModernBERT patient embeddings

These embeddings are generated without supervised fitting. The legacy supervised head consumes them without feature standardization; all classifier fitting and epoch selection use outer-training patients only.


In [ ]:
if EMBEDDING_NPZ_PATH.is_file() and not FORCE_RECOMPUTE_EMBEDDINGS:
    print('Reusing:', EMBEDDING_NPZ_PATH)
else:
    subprocess.run(
        [
            sys.executable, str(PROJECT_DIR / 'prepare_patient_embeddings.py'),
            '--mapping-path', str(MAPPING_PATH),
            '--source-path', str(SOURCE_PATH),
            '--output-dir', str(EMBEDDING_OUTPUT_DIR),
            '--batch-size', str(EMBEDDING_BATCH_SIZE),
            '--device', 'cuda',
        ],
        check=True,
    )
if not EMBEDDING_NPZ_PATH.is_file():
    raise FileNotFoundError(f'Embedding arrays were not created: {EMBEDDING_NPZ_PATH}')


In [ ]:
with np.load(EMBEDDING_NPZ_PATH, allow_pickle=False) as saved:
    patient_ids = saved['patient_ids']
    X = saved['X']
    y = saved['y']
    label_names = saved['label_names'].astype(str)

assert len(patient_ids) == EXPECTED_PATIENTS
assert len(np.unique(patient_ids)) == EXPECTED_PATIENTS
assert X.shape == (EXPECTED_PATIENTS, 768)
assert y.shape == (EXPECTED_PATIENTS, 5)
assert label_names.tolist() == ['dysp', 'cough', 'pain', 'fever', 'nasal']
print('Validated embedding arrays:', X.shape, y.shape)


## 3. Confirm the immutable experiment settings

The outer split is five-fold patient-level multilabel stratification. Every outer fold has 8,000 training-pool and 2,000 unseen patients. Nested supervision subsets contain 400, 800, 1,600, and 8,000 patients. A legacy 90/10 training/validation split selects the duration independently inside each selected outer-training subset; the selected duration is then refit on that complete subset.


In [ ]:
import importlib
sys.path.insert(0, str(PROJECT_DIR))
importlib.invalidate_caches()
for module_name in ['legacy_oof_symptom_classifier', 'run_oof_graph_experiment']:
    sys.modules.pop(module_name, None)
from legacy_oof_symptom_classifier import ClassifierConfig
from run_oof_graph_experiment import PCConfig, SUPERVISION_FRACTIONS
print('Loaded graph code from:', sys.modules['run_oof_graph_experiment'].__file__)
print('Repository commit:', subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True
).strip())

classifier_config = ClassifierConfig()
pc_config = PCConfig()
assert SUPERVISION_FRACTIONS == (0.05, 0.10, 0.20, 1.00)
assert classifier_config.implementation == 'legacy_reproduction'
assert classifier_config.model_class == 'LegacyHeadOnlyModel'
assert classifier_config.training_seed == 5
assert classifier_config.head_dim == 256
assert classifier_config.learning_rate == 3e-5
assert classifier_config.feature_standardization is False
assert classifier_config.threshold_operator == '>'
assert pc_config.algorithm == 'PC-learn'
assert pc_config.implementation == 'pgmpy'
assert pc_config.model_class == 'DiscreteBayesianNetwork'
assert pc_config.conditional_independence_test == 'g_sq'
assert pc_config.alpha == 0.05
assert pc_config.stable is True
assert pc_config.return_type == 'dag'
assert pc_config.max_k is None
print('Classifier:', json.dumps(classifier_config.to_dict(), indent=2))
print('PC-learn:', json.dumps(pc_config.__dict__, indent=2))


## 4. Run lightweight-head OOF training, symptom metrics, and PC-learn

This is the long-running cell. It trains 20 final legacy outer models (four supervision levels × five outer folds), with legacy validation-loss epoch selection inside each training subset. Results and checkpoints are written directly to the legacy-specific Drive directory. After the four complete OOF matrices are assembled, PC-learn runs once per reconstructed 10,000-row dataset and once for the oracle.


In [ ]:
GRAPH_METRICS_PATH = OOF_OUTPUT_DIR / 'graph_metrics.csv'
GRAPH_CONFIG_PATH = OOF_OUTPUT_DIR / 'pc_learn_config.json'
EXPERIMENT_METADATA_PATH = OOF_OUTPUT_DIR / 'experiment_metadata.json'
saved_graph_config = {}
if GRAPH_CONFIG_PATH.is_file():
    try:
        saved_graph_config = json.loads(GRAPH_CONFIG_PATH.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        saved_graph_config = {}
saved_experiment_metadata = {}
if EXPERIMENT_METADATA_PATH.is_file():
    try:
        saved_experiment_metadata = json.loads(
            EXPERIMENT_METADATA_PATH.read_text(encoding='utf-8')
        )
    except (OSError, json.JSONDecodeError):
        saved_experiment_metadata = {}
outputs_match_current_backend = (
    saved_graph_config.get('implementation') == 'pgmpy'
    and saved_graph_config.get('model_class') == 'DiscreteBayesianNetwork'
    and saved_graph_config.get('conditional_independence_test') == 'g_sq'
    and saved_graph_config.get('return_type') == 'dag'
    and saved_experiment_metadata.get('classifier_backend') == 'legacy'
)
print('Metrics path:', GRAPH_METRICS_PATH)
print('Metrics exists:', GRAPH_METRICS_PATH.is_file())
print('Force rerun:', FORCE_RERUN_OOF_EXPERIMENT)
print('Saved graph backend:', saved_graph_config.get('implementation'))
print('Saved classifier backend:', saved_experiment_metadata.get('classifier_backend'))
if (
    GRAPH_METRICS_PATH.is_file()
    and outputs_match_current_backend
    and not FORCE_RERUN_OOF_EXPERIMENT
):
    print('Complete experiment outputs already exist; reusing:', OOF_OUTPUT_DIR)
else:
    if GRAPH_METRICS_PATH.is_file() and not outputs_match_current_backend:
        print('Existing outputs do not match legacy classifier + pgmpy DAG; rerunning.')
    command = [
        sys.executable, '-u', str(PROJECT_DIR / 'run_oof_graph_experiment.py'),
        '--embedding-npz', str(EMBEDDING_NPZ_PATH),
        '--structured-data', str(SOURCE_PATH),
        '--reference-adjacency', str(REFERENCE_ADJACENCY_PATH),
        '--output-dir', str(OOF_OUTPUT_DIR),
        '--patient-id-column', PATIENT_ID_COLUMN,
        '--seed', str(RANDOM_SEED),
        '--device', 'cuda',
        '--classifier-backend', 'legacy',
        '--show-pc-progress',
    ]
    print('Running:', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for output_line in process.stdout:
        print(output_line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f'OOF graph experiment failed with exit code {return_code}. '
            'The complete child-process traceback is printed immediately above.'
        )
if not GRAPH_METRICS_PATH.is_file():
    raise FileNotFoundError(f'Experiment did not finish: {GRAPH_METRICS_PATH}')


## 5. Symptom-prediction results


In [ ]:
classifier_oof_metrics = pd.read_csv(OOF_OUTPUT_DIR / 'classifier_oof_metrics.csv')
classifier_per_symptom_metrics = pd.read_csv(
    OOF_OUTPUT_DIR / 'classifier_per_symptom_metrics.csv'
)
classifier_fold_metrics = pd.read_csv(OOF_OUTPUT_DIR / 'classifier_fold_metrics.csv')

print('Complete 10,000-patient OOF macro metrics')
display(classifier_oof_metrics)
print('Complete OOF per-symptom metrics and fold mean +/- SD')
display(classifier_per_symptom_metrics)
print('Fold-level metrics')
display(classifier_fold_metrics)


## 6. Downstream causal-graph results


In [ ]:
graph_metrics = pd.read_csv(GRAPH_METRICS_PATH)
expected_conditions = ['oracle', 'FS_5', 'FS_10', 'FS_20', 'FS_100']
assert graph_metrics['condition'].tolist() == expected_conditions
assert set(graph_metrics['algorithm']) == {'PC-learn'}
assert set(graph_metrics['conditional_independence_test']) == {'g_sq'}
assert set(graph_metrics['alpha']) == {0.05}
assert set(graph_metrics['stable']) == {True}
assert set(graph_metrics['return_type']) == {'dag'}
display(graph_metrics)

with open(OOF_OUTPUT_DIR / 'pc_learn_config.json', encoding='utf-8') as handle:
    saved_pc_config = json.load(handle)
assert saved_pc_config['implementation'] == 'pgmpy'
assert saved_pc_config['model_class'] == 'DiscreteBayesianNetwork'
assert saved_pc_config['conditional_independence_test'] == 'g_sq'
assert saved_pc_config['return_type'] == 'dag'
assert saved_pc_config['alpha'] == 0.05
with open(OOF_OUTPUT_DIR / 'experiment_metadata.json', encoding='utf-8') as handle:
    saved_experiment_metadata = json.load(handle)
assert saved_experiment_metadata['classifier_backend'] == 'legacy'
assert saved_experiment_metadata['classifier']['model_class'] == 'LegacyHeadOnlyModel'
assert saved_experiment_metadata['classifier']['threshold_operator'] == '>'
print('Saved PC configuration:', json.dumps(saved_pc_config, indent=2))


## 7. Final leakage and artifact audit


In [ ]:
fold_assignments = pd.read_csv(OOF_OUTPUT_DIR / 'outer_fold_assignments.csv')
assert len(fold_assignments) == EXPECTED_PATIENTS
assert fold_assignments['patient_id'].is_unique
assert fold_assignments['outer_fold'].value_counts().sort_index().tolist() == [2000] * 5
assignment_by_id = fold_assignments.set_index('patient_id')['outer_fold']

for suffix in ['005', '010', '020', '100']:
    oof = pd.read_csv(OOF_OUTPUT_DIR / f'oof_predictions_{suffix}.csv')
    assert len(oof) == EXPECTED_PATIENTS
    assert oof['patient_id'].is_unique
    assert set(oof['patient_id']) == set(fold_assignments['patient_id'])
    expected_folds = oof['patient_id'].map(assignment_by_id).to_numpy()
    assert np.array_equal(oof['outer_fold'].to_numpy(), expected_folds)
    for label in ['dysp', 'cough', 'pain', 'fever', 'nasal']:
        assert oof[f'{label}_probability'].between(0, 1).all()
        assert set(oof[f'{label}_prediction'].unique()).issubset({0, 1})

required_artifacts = [
    'classifier_fold_metrics.csv', 'classifier_oof_metrics.csv',
    'classifier_per_symptom_metrics.csv', 'graph_metrics.csv',
    'experiment_metadata.json', 'pc_learn_config.json',
]
for suffix in ['oracle', '005', '010', '020', '100']:
    required_artifacts.extend([f'graph_edges_{suffix}.csv', f'graph_adjacency_{suffix}.csv'])
missing = [name for name in required_artifacts if not (OOF_OUTPUT_DIR / name).is_file()]
assert not missing, f'Missing artifacts: {missing}'
print('PASS: OOF uniqueness, common folds, probability ranges, and graph artifacts validated.')


In [ ]:
# Compact artifact inventory. Everything remains stored in Google Drive.
artifact_rows = [
    {'file': path.name, 'size_mb': round(path.stat().st_size / (1024 ** 2), 3)}
    for path in sorted(OOF_OUTPUT_DIR.glob('*'))
    if path.is_file()
]
display(pd.DataFrame(artifact_rows))
